# Wildfire Ignition Prediction — Exploratory Data Analysis

This notebook explores `feature_df`, the window-level (75-day, pre-event 60-day summarized) wildfire dataset,
after cleaning (sentinel removal, duplicate/conflict window removal) and feature engineering
(weather summary stats, population density, land cover, holiday flags).

**Run this after your main pipeline notebook**, so `feature_df` is already in memory — or load a saved copy below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# If feature_df is not already in memory, uncomment and load your saved copy:
# feature_df = pd.read_parquet('feature_df.parquet')

print(feature_df.shape)
feature_df.head()


## 1. Class Balance

How many windows ended in a wildfire ignition vs. not?

In [ ]:
label_counts = feature_df['label'].value_counts(normalize=True).rename({0: 'No Fire', 1: 'Fire'})
print(label_counts)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=feature_df, x='label', ax=ax)
ax.set_xticklabels(['No Fire', 'Fire'])
ax.set_title('Window-Level Class Balance')
ax.set_xlabel('')
for i, v in enumerate(feature_df['label'].value_counts().sort_index()):
    ax.text(i, v + 500, f'{v:,}', ha='center')
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=150)
plt.show()


## 2. Key Weather Feature Distributions by Outcome

Comparing pre-event weather summary stats between fire and no-fire windows.
Classic fire-weather drivers: dryness (precipitation), fuel moisture, burning index, vapor pressure deficit.

In [ ]:
weather_features_to_check = [
    'precip_mm_mean', 'fuel_moist_100hr_pct_mean', 'burning_index_mean',
    'vpd_kpa_mean', 'temp_max_c_mean', 'rh_min_pct_mean'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, weather_features_to_check):
    sns.kdeplot(data=feature_df, x=col, hue='label', ax=ax, common_norm=False, fill=True, alpha=0.4)
    ax.set_title(col)
    ax.get_legend().remove() if ax.get_legend() else None

handles = [plt.Line2D([0], [0], color=sns.color_palette()[0], lw=4),
           plt.Line2D([0], [0], color=sns.color_palette()[1], lw=4)]
fig.legend(handles, ['No Fire', 'Fire'], loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Pre-Event Weather Feature Distributions by Fire Outcome', y=1.05)
plt.tight_layout()
plt.savefig('eda_weather_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Population Density by Fire Outcome

The core human-activity feature. Population density is heavily right-skewed
(most windows are in remote/rural areas), so a log scale is used.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=feature_df, x='label', y='pop_density', ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_xticklabels(['No Fire', 'Fire'])
axes[0].set_ylabel('Population Density (log scale, per km\u00b2)')
axes[0].set_title('Population Density by Fire Outcome')

sns.violinplot(data=feature_df, x='label', y='pop_density', ax=axes[1], cut=0)
axes[1].set_yscale('log')
axes[1].set_xticklabels(['No Fire', 'Fire'])
axes[1].set_ylabel('Population Density (log scale, per km\u00b2)')
axes[1].set_title('Population Density Distribution by Fire Outcome')

plt.tight_layout()
plt.savefig('eda_pop_density_by_outcome.png', dpi=150)
plt.show()

print(feature_df.groupby('label')['pop_density'].describe())


**Finding:** median population density is roughly 3x higher in fire-outcome windows than
non-fire windows. This foreshadows the permutation importance / SHAP result that population density
is a meaningfully predictive feature for tree-based models, despite showing weak signal in a linear model.

## 4. Land Cover Composition by Fire Outcome

Does land cover type differ between fire and non-fire windows?

In [ ]:
lc_cols = [c for c in feature_df.columns if c.startswith('lc_')]

lc_by_label = feature_df.groupby('label')[lc_cols].mean().T
lc_by_label.columns = ['No Fire', 'Fire']

fig, ax = plt.subplots(figsize=(9, 6))
lc_by_label.plot(kind='barh', ax=ax)
ax.set_xlabel('Proportion of Windows')
ax.set_title('Land Cover Category Prevalence by Fire Outcome')
plt.tight_layout()
plt.savefig('eda_landcover_by_outcome.png', dpi=150)
plt.show()

print(lc_by_label)


## 5. Temporal Patterns

Fire seasonality (month) and human-activity-linked temporal features (weekend, holiday).

In [ ]:
feature_df['event_month'] = pd.to_datetime(feature_df['event_date']).dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly_rate = feature_df.groupby('event_month')['label'].mean()
monthly_rate.plot(kind='bar', ax=axes[0], color=sns.color_palette()[1])
axes[0].set_title('Fire Rate by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Proportion of Windows Ending in Fire')

weekend_rate = feature_df.groupby('is_weekend')['label'].mean()
weekend_rate.index = ['Weekday', 'Weekend']
weekend_rate.plot(kind='bar', ax=axes[1], color=sns.color_palette()[2])
axes[1].set_title('Fire Rate: Weekday vs Weekend')
axes[1].set_ylabel('Proportion of Windows Ending in Fire')
axes[1].set_xticklabels(weekend_rate.index, rotation=0)

plt.tight_layout()
plt.savefig('eda_temporal_patterns.png', dpi=150)
plt.show()


## 6. Geographic Distribution

Where are fire vs. non-fire windows located across the US?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sample = feature_df.sample(n=min(20000, len(feature_df)), random_state=42)

sns.scatterplot(
    data=sample, x='longitude', y='latitude', hue='label',
    palette={0: 'steelblue', 1: 'crimson'}, alpha=0.3, s=8, ax=ax
)
ax.set_title('Geographic Distribution of Sampled Windows (n=%d)' % len(sample))
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['No Fire', 'Fire'], markerscale=3, title='Outcome')
plt.tight_layout()
plt.savefig('eda_geographic_distribution.png', dpi=150)
plt.show()


## 7. Correlation Structure (Final Selected Features)

Correlation heatmap on the 46 features that survived permutation-importance-based selection.
Compare this to the raw, sentinel-corrupted correlation matrix from earlier cleaning
(near-uniform ~0.999 correlations across unrelated variables) — this version should show only
physically sensible relationships (e.g. evapotranspiration variants, fuel moisture timelags).

In [ ]:
# selected_features should be defined from your feature selection step
corr = feature_df[selected_features].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True, linewidths=0.1,
            cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Correlation Matrix — Final 46 Selected Features (Clean Data)')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap_clean.png', dpi=150)
plt.show()


## Summary of EDA Findings

- Class balance: ~26% of windows end in ignition, ~74% do not (measured directly from cleaned data;
  differs from the source paper's implied ~40/60, investigated separately and confirmed not an artifact
  of this project's cleaning).
- Weather features (precipitation, fuel moisture, burning index, VPD) show visibly separated distributions
  between fire and no-fire windows, consistent with known fire-weather drivers.
- Population density is meaningfully higher in fire-outcome windows (median ~3x higher), foreshadowing
  its strong permutation importance and SHAP contribution in tree-based models later in the pipeline.
- Land cover composition differs modestly between classes but was found to carry little independent
  predictive signal once weather and population features were included (see feature selection / SHAP results).
- Fire windows show seasonal concentration, consistent with expected wildfire season patterns.
- The correlation matrix on final selected features shows only physically explainable relationships
  (e.g. actual vs. potential evapotranspiration, fuel moisture across timelags) — confirming the earlier
  sentinel-value data corruption (which produced spurious ~0.999 correlations across unrelated variables)
  was fully resolved before this feature set was finalized.


---
## 8. Deep Dive: Population Density

Population density is the project's core human-activity finding, so it gets a dedicated section
exploring its distribution, its relationship to fire risk directly (not just via the model),
and how it interacts with other features and geography.

### 8.1 Overall Distribution

How is population density distributed across the whole dataset, regardless of outcome?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(feature_df['pop_density'], bins=60, ax=axes[0], color=sns.color_palette()[0])
axes[0].set_title('Population Density Distribution (raw scale)')
axes[0].set_xlabel('Population Density (per km\u00b2)')

sns.histplot(np.log1p(feature_df['pop_density']), bins=60, ax=axes[1], color=sns.color_palette()[3])
axes[1].set_title('Population Density Distribution (log1p scale)')
axes[1].set_xlabel('log(1 + Population Density)')

plt.tight_layout()
plt.savefig('eda_popdensity_overall_dist.png', dpi=150)
plt.show()

print(feature_df['pop_density'].describe())
print('Skewness:', feature_df['pop_density'].skew())


**Note:** the raw distribution is extremely right-skewed (most windows sit in near-zero density
wilderness, with a long tail into dense suburban/urban values) — this is exactly why the SHAP and
boxplot visuals earlier used a log scale, and why `log1p` is a natural transform to consider if
population density is ever fed into a linear model.

### 8.2 Fire Rate by Population Density Bin

Rather than just comparing means/medians between two groups, binning population density into
deciles and plotting the fire rate per bin shows the *shape* of the relationship directly —
is it linear, a threshold effect, or something else?

In [ ]:
feature_df['pop_density_bin'] = pd.qcut(feature_df['pop_density'], q=10, duplicates='drop')

bin_fire_rate = feature_df.groupby('pop_density_bin', observed=True)['label'].agg(['mean', 'count'])
bin_fire_rate['bin_midpoint'] = [interval.mid for interval in bin_fire_rate.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(len(bin_fire_rate)), bin_fire_rate['mean'], marker='o', linewidth=2, color=sns.color_palette()[1])
ax.set_xticks(range(len(bin_fire_rate)))
ax.set_xticklabels([f'{interval.left:.1f}\n-{interval.right:.1f}' for interval in bin_fire_rate.index],
                    rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Population Density Decile (per km\u00b2)')
ax.set_ylabel('Fire Rate (Proportion of Windows)')
ax.set_title('Fire Rate by Population Density Decile')
ax.axhline(feature_df['label'].mean(), color='gray', linestyle='--', alpha=0.6, label='Overall average fire rate')
ax.legend()
plt.tight_layout()
plt.savefig('eda_popdensity_fire_rate_by_decile.png', dpi=150)
plt.show()

print(bin_fire_rate)


**This is the single clearest population-density plot for a writeup.** If the relationship is
monotonic (fire rate climbs steadily decile over decile), it strongly supports a direct dose-response
story. If it plateaus or peaks in the middle deciles (moderate/suburban density) and drops at the very
top (dense urban core), that would support the wildland-urban-interface-specific mechanism discussed
earlier — worth checking the shape carefully rather than assuming it's linear.

### 8.3 Population Density vs. a Key Weather Driver (Interaction Check)

Since permutation importance/SHAP suggested population density's effect is interaction-driven (only visible to non-linear models), a 2D scatter against a top weather feature, colored by outcome, is a natural way to visualize a possible interaction.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sample = feature_df.sample(n=min(15000, len(feature_df)), random_state=42)

sns.scatterplot(
    data=sample, x='precip_mm_mean', y='pop_density', hue='label',
    palette={0: 'steelblue', 1: 'crimson'}, alpha=0.35, s=12, ax=ax
)
ax.set_yscale('log')
ax.set_xlabel('Mean Precipitation, Pre-Event 60 Days (mm/day)')
ax.set_ylabel('Population Density (log scale, per km\u00b2)')
ax.set_title('Population Density vs. Precipitation, by Fire Outcome')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['No Fire', 'Fire'], markerscale=2, title='Outcome')
plt.tight_layout()
plt.savefig('eda_popdensity_vs_precip_interaction.png', dpi=150)
plt.show()


**What to look for:** if fire points (red) are concentrated specifically in the
*low-precipitation + moderate-to-high population density* corner of this plot — rather than just
being spread evenly along the population density axis regardless of precipitation — that's visual
evidence of the interaction effect the model is exploiting: dry conditions AND human presence
together, not either alone.

### 8.4 Population Density by Land Cover Category

Does population density itself vary meaningfully by land cover type? This checks whether land cover's weak standalone importance (Section 4, and the SHAP/permutation importance results) might partly be because population density already captures similar information.

In [ ]:
lc_cols = [c for c in feature_df.columns if c.startswith('lc_')]

# reconstruct a single land-cover-group label per row from the one-hot columns for plotting
lc_group = feature_df[lc_cols].idxmax(axis=1).str.replace('lc_', '', regex=False)

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = feature_df.assign(land_cover_group=lc_group)
order = plot_df.groupby('land_cover_group')['pop_density'].median().sort_values(ascending=False).index

sns.boxplot(data=plot_df, x='land_cover_group', y='pop_density', order=order, ax=ax)
ax.set_yscale('log')
ax.set_xlabel('Land Cover Group')
ax.set_ylabel('Population Density (log scale, per km\u00b2)')
ax.set_title('Population Density by Land Cover Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('eda_popdensity_by_landcover.png', dpi=150)
plt.show()


### 8.5 Population Density Geographically

Mapping population density directly (not just fire outcome) shows where the WUI-type areas actually are, which helps sanity-check the population/land-cover join worked correctly and shows the spatial texture of the feature.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sample = feature_df.sample(n=min(20000, len(feature_df)), random_state=42)

sc = ax.scatter(
    sample['longitude'], sample['latitude'],
    c=np.log1p(sample['pop_density']), cmap='viridis', s=8, alpha=0.5
)
plt.colorbar(sc, ax=ax, label='log(1 + Population Density)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic Distribution of Population Density (Sampled Windows, n=%d)' % len(sample))
plt.tight_layout()
plt.savefig('eda_popdensity_geographic.png', dpi=150)
plt.show()


### Section Summary

- Population density is heavily right-skewed; most windows sit in near-zero-density wilderness,
  with a long tail into suburban/urban values.
- Binning into deciles shows the actual shape of the fire-rate relationship directly from the raw
  data — worth checking whether it's monotonic or peaks at moderate (WUI-range) density before
  dropping off at the highest urban densities.
- The precipitation interaction scatter is a visual, non-model-based check on whether dry conditions
  combined with human presence specifically differentiate fire windows — supporting evidence for
  why tree-based models (which can learn interactions) found this feature far more useful than
  logistic regression did.
- Land cover and population density show some relationship to each other, which may partly explain
  why land cover carried little independent signal once population density was included as a feature.
